# Combined Regression Across All Team Datasets

This notebook pools all `_day_night.csv` team datasets into one combined DataFrame and runs the cleaned regression analysis on the full set of games together.

It saves pooled outputs to `analysis/cleaned_analysis/ALL_TEAMS/`.

By default, this is a pooled model without team fixed effects. The goal is to answer the broad question: how do weather, day/night, and wind variables relate to outcomes across all available stadium datasets combined?

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from matplotlib.colors import Normalize, TwoSlopeNorm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

# ============================================================
# PARAMETERS
# ============================================================
SEASON_START = None
SEASON_END = None

DATASETS_DIR = os.path.join('..', 'data')
OUTPUT_DIR = os.path.join('cleaned_analysis', 'ALL_TEAMS')
SUMMARY_PATH = os.path.join(OUTPUT_DIR, 'combined_regression_summary.csv')

params_df = pd.read_csv('team_parameters.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Output dir: {os.path.abspath(OUTPUT_DIR)}')

In [ ]:
# ============================================================
# MODEL / PLOT CONFIGURATION
# ============================================================
DEPENDENT_VARS = {
    'away_runs_scored': 'Away Runs Scored',
    'away_bat_k': 'Away Strikeouts',
    'away_bat_hr_h_ratio': 'Away HR:H Ratio',
}

MODEL_SPECS = {
    'with_wind': {
        'label': 'With Wind + Day/Night Feature',
        'ivars': ['temp_f', 'rhum', 'pres', 'prcp', 'is_night', 'wspd_mph', 'wind_cf', 'wind_lcf', 'wind_rcf'],
        'center_vars': ['temp_f', 'rhum', 'pres', 'prcp', 'wspd_mph', 'wind_cf', 'wind_lcf', 'wind_rcf'],
    },
    'without_wind': {
        'label': 'Without Wind + Day/Night Feature',
        'ivars': ['temp_f', 'rhum', 'pres', 'prcp', 'is_night'],
        'center_vars': ['temp_f', 'rhum', 'pres', 'prcp'],
    },
}

IV_DISPLAY_NAMES = {
    'temp_f': 'Temp (F)',
    'rhum': 'Humidity (%)',
    'pres': 'Pressure (hPa)',
    'prcp': 'Precip (mm)',
    'wspd_mph': 'Wind Speed (mph)',
    'wind_cf': 'Wind to CF',
    'wind_lcf': 'Wind to LCF',
    'wind_rcf': 'Wind to RCF',
    'is_night': 'Night Game (1=yes)',
    'const': 'Intercept',
}

WEATHER_VARS = {
    'temp_f': {'label': 'Temperature', 'unit': '°F', 'n_bins': 5, 'fmt': '.0f'},
    'wspd_mph': {'label': 'Wind Speed', 'unit': ' mph', 'n_bins': 5, 'fmt': '.0f'},
    'rhum': {'label': 'Humidity', 'unit': '%', 'n_bins': 5, 'fmt': '.0f'},
    'pres': {'label': 'Pressure', 'unit': ' hPa', 'n_bins': 5, 'fmt': '.0f'},
}

BASEBALL_STATS = {
    'away_runs_scored': {'label': 'Away Runs', 'fmt': '.1f'},
    'away_bat_hr': {'label': 'Away Home Runs', 'fmt': '.2f'},
    'away_bat_k': {'label': 'Away Strikeouts', 'fmt': '.1f'},
    'away_bat_bb': {'label': 'Away Walks', 'fmt': '.1f'},
    'away_bat_hr_h_ratio': {'label': 'Away HR:H Ratio', 'fmt': '.3f'},
}

WIND_PROJ_VARS = {
    'wind_cf': {'label': 'Wind to CF', 'color': '#c0392b'},
    'wind_lcf': {'label': 'Wind to LCF', 'color': '#2980b9'},
    'wind_rcf': {'label': 'Wind to RCF', 'color': '#27ae60'},
}

MIN_GAMES_PER_BIN = 20
N_WIND_BINS = 12

print('Configuration loaded.')

In [ ]:
def day_night_dataset_file(dataset_file):
    base, ext = os.path.splitext(dataset_file)
    return f'{base}_day_night{ext}'


def load_combined_data(params_df):
    frames = []
    for _, row in params_df.iterrows():
        dataset_file = day_night_dataset_file(row['dataset_file'])
        dataset_path = os.path.join(DATASETS_DIR, dataset_file)
        if not os.path.exists(dataset_path):
            print(f'Skipping missing dataset: {dataset_file}')
            continue

        df = pd.read_csv(dataset_path)
        df['game_date'] = pd.to_datetime(df['game_date'])
        df['home_team'] = row['team_code']
        df['stadium_name'] = row['stadium_name']

        if SEASON_START is not None:
            df = df[df['season'] >= int(SEASON_START)]
        if SEASON_END is not None:
            df = df[df['season'] <= int(SEASON_END)]

        frames.append(df)

    if not frames:
        raise ValueError('No _day_night datasets were found to combine.')

    combined = pd.concat(frames, ignore_index=True)
    if 'day_night' not in combined.columns:
        if 'start_hour' not in combined.columns:
            raise ValueError('Combined data is missing both day_night and start_hour.')
        combined['day_night'] = np.where(pd.to_numeric(combined['start_hour'], errors='coerce') < 17, 'day', 'night')

    combined['day_night'] = combined['day_night'].astype(str).str.strip().str.lower()
    combined.loc[~combined['day_night'].isin(['day', 'night']), 'day_night'] = pd.NA
    combined['is_night'] = np.where(combined['day_night'] == 'night', 1.0, 0.0)
    combined.loc[combined['day_night'].isna(), 'is_night'] = np.nan
    return combined


def run_ols(df, dep_var, indep_vars, center_vars):
    cols_needed = [dep_var] + indep_vars
    reg_df = df[cols_needed].dropna().copy()
    n_dropped = len(df) - len(reg_df)
    if len(reg_df) == 0:
        raise ValueError('No rows remain after dropping NA values for regression')

    for col in center_vars:
        reg_df[col] = reg_df[col] - reg_df[col].mean()

    X = sm.add_constant(reg_df[indep_vars])
    y = reg_df[dep_var]
    model = sm.OLS(y, X).fit()
    return model, n_dropped


def results_to_dataframe(model):
    summary_df = pd.DataFrame({
        'Variable': [IV_DISPLAY_NAMES.get(v, v) for v in model.params.index],
        'Coefficient': model.params.values,
        'Std Error': model.bse.values,
        't-stat': model.tvalues.values,
        'P-value': model.pvalues.values,
    })

    def sig_stars(p):
        if p < 0.001:
            return '***'
        if p < 0.01:
            return '**'
        if p < 0.05:
            return '*'
        if p < 0.10:
            return '.'
        return ''

    summary_df['Sig'] = summary_df['P-value'].apply(sig_stars)
    summary_df['Coefficient'] = summary_df['Coefficient'].round(4)
    summary_df['Std Error'] = summary_df['Std Error'].round(4)
    summary_df['t-stat'] = summary_df['t-stat'].round(3)
    summary_df['P-value'] = summary_df['P-value'].round(4)
    return summary_df


def render_regression_table(results_df, model, dep_label, model_label, n_obs, n_dropped, filepath, show=True):
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.axis('off')
    ax.set_title(f'Pooled OLS Regression: {dep_label}\nAll Team Datasets Combined — {model_label}', fontsize=13, fontweight='bold', pad=20, loc='left')

    table = ax.table(cellText=results_df.values.tolist(), colLabels=results_df.columns.tolist(), cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.45)

    for j in range(len(results_df.columns)):
        table[0, j].set_facecolor('#2c3e50')
        table[0, j].set_text_props(color='white', fontweight='bold')

    for i in range(1, len(results_df) + 1):
        for j in range(len(results_df.columns)):
            if i % 2 == 0:
                table[i, j].set_facecolor('#f0f0f0')

    intercept_val = model.params.get('const', np.nan)
    summary_text = (
        f'n = {n_obs}    '
        f'R² = {model.rsquared:.4f}    '
        f'Adj R² = {model.rsquared_adj:.4f}    '
        f'F = {model.fvalue:.2f} (p = {model.f_pvalue:.4f})    '
        f'Intercept = {intercept_val:.2f}'
    )
    if n_dropped > 0:
        summary_text += f'    [{n_dropped} obs dropped]'

    fig.text(0.05, 0.02, summary_text, fontsize=9, fontstyle='italic', color='#555555')
    fig.text(0.95, 0.02, 'Sig: *** p<0.001  ** p<0.01  * p<0.05  . p<0.10', fontsize=8, ha='right', color='#888888')
    plt.tight_layout()
    fig.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)


def bucket_weather_var(df, col, config):
    series = df[col].dropna()
    q = min(config['n_bins'], max(1, series.nunique()))
    if len(series) == 0 or q < 2:
        return None, [], {}
    buckets = pd.qcut(series, q=q, duplicates='drop')
    label_map = {}
    for interval in buckets.cat.categories:
        left = format(interval.left, config['fmt'])
        right = format(interval.right, config['fmt'])
        label_map[interval] = f'{left}-{right}{config["unit"]}'
    labeled = buckets.map(label_map)
    bucket_order = [label_map[iv] for iv in buckets.cat.categories]
    counts = labeled.value_counts().reindex(bucket_order).fillna(0).astype(int).to_dict()
    return labeled, bucket_order, counts


def render_heatmap(df, weather_col, weather_config, version, filepath, show=True):
    labeled, bucket_order, counts = bucket_weather_var(df, weather_col, weather_config)
    if labeled is None or len(bucket_order) == 0:
        return None

    stat_keys = list(BASEBALL_STATS.keys())
    stat_labels = [BASEBALL_STATS[k]['label'] for k in stat_keys]
    df_valid = df.loc[labeled.index].copy()
    df_valid['_bucket'] = labeled.values

    matrix = np.full((len(stat_keys), len(bucket_order)), np.nan)
    for i, stat in enumerate(stat_keys):
        overall_mean = df[stat].mean()
        grouped = df_valid.groupby('_bucket')[stat].mean()
        for j, bucket_label in enumerate(bucket_order):
            if bucket_label in grouped.index:
                val = grouped[bucket_label]
                matrix[i, j] = val if version == 'raw' else val - overall_mean

    if np.isnan(matrix).all():
        return None

    cmap = plt.cm.YlOrRd if version == 'raw' else plt.cm.RdBu_r
    finite_vals = matrix[np.isfinite(matrix)]
    if version == 'raw':
        norm = Normalize(vmin=np.nanmin(finite_vals), vmax=np.nanmax(finite_vals))
    else:
        bound = max(abs(np.nanmin(finite_vals)), abs(np.nanmax(finite_vals)))
        norm = TwoSlopeNorm(vmin=-bound, vcenter=0, vmax=bound)

    fig, ax = plt.subplots(figsize=(max(8, len(bucket_order) * 1.8), 5))
    im = ax.imshow(matrix, cmap=cmap, norm=norm, aspect='auto')
    ax.set_xticks(np.arange(len(bucket_order)))
    ax.set_yticks(np.arange(len(stat_labels)))
    ax.set_xticklabels([f'{label}\n(n={counts.get(label, 0)})' for label in bucket_order], rotation=0, fontsize=9)
    ax.set_yticklabels(stat_labels, fontsize=10)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if np.isfinite(matrix[i, j]):
                ax.text(j, i, f'{matrix[i, j]:.2f}', ha='center', va='center', fontsize=9, color='black')

    version_label = 'Raw Mean' if version == 'raw' else 'Difference from Combined Mean'
    ax.set_title(f'Combined {weather_config["label"]} Heatmap ({version_label})', fontsize=13)
    cbar = fig.colorbar(im, ax=ax, shrink=0.9)
    cbar.set_label(weather_config['label'])
    plt.tight_layout()
    fig.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return filepath


def render_wind_line_graph(df, stat_col, stat_config, shared_bins, filepath, show=True):
    fig, ax = plt.subplots(figsize=(10, 6))
    plotted = False
    for wind_col, wind_config in WIND_PROJ_VARS.items():
        series = df[[wind_col, stat_col]].dropna().copy()
        if len(series) == 0:
            continue
        series['_bin'] = pd.cut(series[wind_col], bins=shared_bins, include_lowest=True)
        grouped = series.groupby('_bin')[stat_col].agg(['mean', 'count'])
        grouped = grouped[grouped['count'] >= MIN_GAMES_PER_BIN]
        if len(grouped) == 0:
            continue
        midpoints = [(iv.left + iv.right) / 2 for iv in grouped.index]
        ax.plot(midpoints, grouped['mean'].values, marker='o', markersize=5, linewidth=2, color=wind_config['color'], label=wind_config['label'])
        plotted = True

    if not plotted:
        plt.close(fig)
        return None

    ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Wind Projection (mph) — negative = blowing in, positive = blowing out', fontsize=12)
    ax.set_ylabel(stat_config['label'], fontsize=12)
    ax.set_title(f'Combined {stat_config["label"]} by Wind Projection', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return filepath


print('Helper functions defined.')

In [ ]:
combined = load_combined_data(params_df)
print(f'Combined rows: {len(combined)}')
print(f'Teams represented: {combined["home_team"].nunique()}')
combined[['game_date', 'season', 'home_team', 'away_team', 'day_night', 'is_night']].head()

In [ ]:
results_summary = []

# 1. Pooled regressions
for dep_var, dep_label in DEPENDENT_VARS.items():
    for model_key, model_spec in MODEL_SPECS.items():
        model, n_dropped = run_ols(combined, dep_var, model_spec['ivars'], model_spec['center_vars'])
        results_df = results_to_dataframe(model)
        filepath = os.path.join(OUTPUT_DIR, f'ols_{model_key}_{dep_var}.png')
        render_regression_table(results_df, model, dep_label, model_spec['label'], int(model.nobs), n_dropped, filepath, show=True)
        results_summary.append({
            'dependent_var': dep_var,
            'model': model_key,
            'n_obs': int(model.nobs),
            'n_dropped': int(n_dropped),
            'r_squared': round(float(model.rsquared), 4),
            'adj_r_squared': round(float(model.rsquared_adj), 4),
            'f_pvalue': round(float(model.f_pvalue), 6),
            'output_file': os.path.basename(filepath),
        })

# 2. Combined heatmaps
for weather_col, weather_config in WEATHER_VARS.items():
    for version in ['raw', 'diff']:
        filepath = os.path.join(OUTPUT_DIR, f'heatmap_{weather_col}_{version}.png')
        render_heatmap(combined, weather_col, weather_config, version, filepath, show=True)

# 3. Combined wind line graphs
all_wind_vals = pd.concat([combined[c].dropna() for c in WIND_PROJ_VARS if c in combined.columns], ignore_index=True)
if len(all_wind_vals) > 0:
    shared_bins = np.linspace(all_wind_vals.min(), all_wind_vals.max(), N_WIND_BINS + 1)
    for stat_col, stat_config in BASEBALL_STATS.items():
        filepath = os.path.join(OUTPUT_DIR, f'wind_projection_{stat_col}.png')
        render_wind_line_graph(combined, stat_col, stat_config, shared_bins, filepath, show=True)

summary_df = pd.DataFrame(results_summary)
summary_df.to_csv(SUMMARY_PATH, index=False)
print(f'Saved summary: {os.path.abspath(SUMMARY_PATH)}')
summary_df